# Practical P33: Security Audit & Hardening Your Scripts
**Learning Outcome**: Audit and harden AI Python projects against credential exposure.

## Part 1: Writing `security_check.py`
Let's programmatically write a script that scans file directories for pattern signs indicating hardcoded secrets.


In [ ]:
%%writefile security_check.py
import os
import re

def scan_file_for_secrets(filepath):
    # Matches standard patterns like sk- followed by alphanumeric characters, or API_KEY =
    secret_pattern = re.compile(r'(sk-[a-zA-Z0-9]{20,}|API_KEY\s*=\s*["\'][^"\']{10,}["\'])')
    
    findings = []
    with open(filepath, 'r', errors='ignore') as f:
        for idx, line in enumerate(f):
            match = secret_pattern.search(line)
            if match:
                findings.append((idx + 1, match.group(0)))
    return findings

def run_audit(directory='.'):
    print('Beginning Security Audit for credentials exposure...')
    found_any = False
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.py') or file.endswith('.env') or file.endswith('.json'):
                if 'security_check.py' in file: continue
                if '.env.example' in file: continue
                path = os.path.join(root, file)
                matches = scan_file_for_secrets(path)
                if matches:
                    found_any = True
                    print(f'[WARNING] Hardcoded credential found in {path}:')
                    for line_num, secret in matches:
                        # Mask secret for log display
                        masked = secret[:6] + '*' * 10
                        print(f' - Line {line_num}: {masked}')
    if not found_any:
        print('[SUCCESS] No hardcoded credentials detected in directories.')

if __name__ == '__main__':
    run_audit()


In [ ]:
!python3 security_check.py
